**[🏠 Course Home](00_START_HERE.ipynb)** | **🤖 [AI-Native](../ai/04_calculating_a_mean_bayesian.ipynb)** | **📖 [Conceptual](../conceptual/04_calculating_a_mean_bayesian.ipynb)** | **🐍 [Python](../python/04_calculating_a_mean_bayesian.ipynb)** | **⬅️ [Prev: Ch 03: Calculating a Mean (Frequentist)](03_calculating_a_mean_frequentist.ipynb)** | **➡️ [Next: Ch 05: 2D Grid Approximation](05_frequentist_vs_grid_approximation.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 04 of 22*

---

# Chapter 4: Calculating a Mean in a Bayesian World
### *Priors, Likelihoods, The Normal Conjugate Tug-of-War, MCMC Sampling, and Posterior Predictive Inference*

* **Authoritative Track Reference**: [🐍 Python Lab Companion](../python/04_calculating_a_mean_bayesian.ipynb)
* **Conceptual Companion Guide**: [📖 Conceptual Guide: The Tug-of-War](../conceptual/04_calculating_a_mean_bayesian.ipynb)



## 🧭 Executive Summary: Frequentist vs. Bayesian Mean Estimation at a Glance

| Feature / Dimension | Frequentist Paradigm | Bayesian Paradigm |
| :--- | :--- | :--- |
| **Core Epistemology** | Parameters ($\mu$) are fixed, immutable constants of nature. Probability represents **long-run frequencies** over infinite hypothetical repetitions. | Parameters ($\mu$) are unobserved random variables. Probability represents **degrees of belief / plausibility** conditioned on evidence. |
| **Primary Output** | A single point estimate $\bar{x}$, a standard error $SE$, and a 95% Confidence Interval. | A complete **posterior probability distribution** $p(\mu \mid y)$ capturing all remaining uncertainty. |
| **Point Estimate** | Arithmetic sample mean $\bar{x} = \frac{1}{n}\sum x_i$ (MVUE & MLE). | Depends on the chosen **loss function**: Posterior Mean (squared error), Posterior Median (absolute error), or MAP (0-1 loss). |
| **Interval Meaning** | **95% Confidence Interval**: In 95% of hypothetical repeated identical experiments, the calculated interval covers $\mu$. | **95% Credible Interval (HDI)**: Given the prior and observed data, there is a 95% probability that the true parameter $\mu$ lies inside this exact interval. |
| **Prior Knowledge** | Forbidden / impossible to formally incorporate; all inference must flow strictly from the current sample. | **Natural & explicit**: Prior domain knowledge $p(\mu)$ acts as regularizing evidence, preventing overfitting in small samples. |
| **Small Sample Behavior** | Fragile when $n$ is small ($n < 5$); intervals explode in width or fail under non-normality. | Well-calibrated; informative priors stabilize estimates (e.g. cold-start microservice telemetry). |
| **Handling Outliers** | Arithmetic mean has a 0% breakdown point; single outlier pulls $\bar{x}$ dramatically. | **Robust modeling**: Swap Gaussian likelihood for Student-$t$ likelihood to automatically down-weight outliers. |
| **Future Predictions** | Plug-in prediction intervals substitute point estimates $(\bar{x}, s)$, understating parameter uncertainty. | **Posterior Predictive Distribution**: Integrates over all parameter uncertainty: $p(\tilde{y} \mid y) = \int p(\tilde{y} \mid \mu, \sigma) p(\mu, \sigma \mid y) d\mu d\sigma$. |
| **Decision Rules** | Arbitrary binary threshold ($p < 0.05$); cannot accept the null hypothesis. | Continuous decision analysis via **Region of Practical Equivalence (ROPE)** and direct risk calculation. |



## Part 1: The Production Scenario & Observed Data

### 1.1 The Engineering Context
We examine the exact same production microservice benchmark as our frequentist worksheet to ensure direct comparability:
* We collect $n = 35$ execution latency measurements (in milliseconds) from a core transaction service.
* Ground truth parameters (unknown in production): $\mu = 142.5\text{ ms}$, $\sigma = 28.0\text{ ms}$.
* **Prior Engineering Knowledge**: Historical telemetry from previous versions of the service suggests that mean latency typically hovers around $\mu_0 = 150.0\text{ ms}$ with an uncertainty scale $\sigma_0 = 20.0\text{ ms}$.

Let's load the libraries, generate the observed dataset, and inspect our prior beliefs.



In [ ]:
# Set random seed for exact reproducibility
set.seed(42)

# Ground truth population parameters (hidden from engineer)
true_mu <- 145.0   # True mean latency (ms)
true_sigma <- 15.0 # True population standard deviation (ms)
n <- 50            # Sample size (observed requests)

# Generate synthetic latency measurements from Normal distribution
y <- rnorm(n, mean = true_mu, sd = true_sigma)
y_bar <- mean(y)
s_y <- sd(y)

cat("==================================================
")
cat("     BAYESIAN MEAN ESTIMATION DATASET SUMMARY     
")
cat("==================================================
")
cat(sprintf("Observed Sample Size (n)          : %d requests
", n))
cat(sprintf("Sample Mean (y_bar)               : %.3f ms
", y_bar))
cat(sprintf("Sample Standard Deviation (s)     : %.3f ms
", s_y))
cat(sprintf("Sample Standard Error (SE)        : %.3f ms
", s_y / sqrt(n)))
cat("==================================================
")


---

## Part 2: Route 1 — Exact Analytical Conjugacy (The Gaussian Tug-of-War)

When we assume the measurement noise scale $\sigma$ is known (or approximated by sample standard deviation $s$), estimating a Gaussian mean with a Gaussian prior is an **exact conjugate model**.

### 2.1 Bayes' Theorem for Continuous Parameters
$$p(\mu \mid y) = \frac{p(y \mid \mu) \cdot p(\mu)}{p(y)} \propto p(y \mid \mu) \cdot p(\mu)$$
* **Prior**: $p(\mu) = \mathcal{N}(\mu_0, \sigma_0^2)$
* **Likelihood**: $p(y \mid \mu) = \prod_{i=1}^n \mathcal{N}(y_i \mid \mu, \sigma^2) \propto \mathcal{N}\left(\bar{y} \;\middle|\; \mu, \frac{\sigma^2}{n}\right)$

### 2.2 Precision as Voting Power ($\tau = 1 / \sigma^2$)
In Bayesian statistics, **precision** ($\tau$) is the reciprocal of variance:
$$\tau_0 = \frac{1}{\sigma_0^2} \quad (\text{Prior Precision}), \qquad \tau_n = \frac{n}{\sigma^2} \quad (\text{Data Precision})$$

Because precision measures information density, the posterior precision is simply the **direct sum of precisions**:
$$\tau_{\text{post}} = \tau_0 + \tau_n \implies \sigma_{\text{post}}^2 = \frac{1}{\tau_0 + \tau_n}$$

The posterior mean $\mu_{\text{post}}$ is a precision-weighted compromise (a "Tug-of-War"):
$$\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_n \bar{y}}{\tau_0 + \tau_n} = \left(\frac{\tau_0}{\tau_{\text{post}}}\right)\mu_0 + \left(\frac{\tau_n}{\tau_{\text{post}}}\right)\bar{y}$$

> [!IMPORTANT]
> **The Two Extreme Limits**:
> 1. **Zero Prior Knowledge (Flat / Non-Informative Prior, $\sigma_0 \to \infty \implies \tau_0 \to 0$)**:
>    $$\mu_{\text{post}} \to \bar{y}, \qquad \sigma_{\text{post}} \to \frac{\sigma}{\sqrt{n}} = SE$$
>    Under an uninformative prior, the Bayesian posterior matches the frequentist sampling distribution mathematically, but with a vastly superior epistemological interpretation: you can now legitimately say there is a 95% probability $\mu$ is in the interval!
> 2. **Infinite Data ($n \to \infty \implies \tau_n \to \infty$)**:
>    The data completely overwhelms the prior, regardless of prior strength.



In [ ]:
# Analytical Gaussian Conjugate Updating
sigma_known <- sd(y) # Approximated by sample standard deviation

# 1. Informative Prior (Historical deployment: mu_0 = 150, sigma_0 = 20)
prior_mu <- 150.0
prior_sigma <- 20.0

tau_0 <- 1.0 / (prior_sigma^2)
tau_data <- n / (sigma_known^2)

tau_post <- tau_0 + tau_data
post_var <- 1.0 / tau_post
post_std <- sqrt(post_var)
post_mu <- (tau_0 * prior_mu + tau_data * y_bar) / tau_post

# 2. Vague Prior (mu_0 = 0, sigma_0 = 1000)
tau_0_vague <- 1.0 / (1000.0^2)
tau_post_vague <- tau_0_vague + tau_data
post_std_vague <- sqrt(1.0 / tau_post_vague)
post_mu_vague <- (tau_0_vague * 0.0 + tau_data * y_bar) / tau_post_vague

cat("==================================================
")
cat("     EXACT GAUSSIAN CONJUGATE UPDATING RESULTS    
")
cat("==================================================
")
cat(sprintf("Prior Precision (tau_0)           : %.6f
", tau_0))
cat(sprintf("Data Precision (tau_data)         : %.6f (Weight: %.1f%%)
", tau_data, 100 * tau_data / tau_post))
cat(sprintf("Posterior Precision (tau_post)    : %.6f
", tau_post))
cat(sprintf("Posterior Mean (mu_post)          : %.3f ms
", post_mu))
cat(sprintf("Posterior Std Dev (sigma_post)    : %.3f ms
", post_std))
cat(sprintf("95%% Credible Interval (Conjugate) : [%.3f, %.3f] ms
", 
            post_mu - 1.96 * post_std, post_mu + 1.96 * post_std))
cat("--------------------------------------------------
")
cat(sprintf("Vague Prior Posterior Mean        : %.3f ms (y_bar = %.3f ms)
", post_mu_vague, y_bar))
cat(sprintf("Vague Prior Posterior Std Dev     : %.3f ms (SE = %.3f ms)
", post_std_vague, sigma_known / sqrt(n)))
cat("==================================================
")


---

## Part 3: Unknown Variance — The Joint Posterior $p(\mu, \sigma \mid y)$ & MCMC Sampling

In real systems, the noise scale $\sigma$ is **never known exactly**. We must estimate $\mu$ and $\sigma$ simultaneously as joint parameters $\theta = (\mu, \sigma)$.

### 3.1 The Joint Model
* **Parameter Space**: $\mu \in (-\infty, \infty), \quad \sigma \in (0, \infty)$ (parametrized in log-space: $\log\sigma \in (-\infty, \infty)$)
* **Priors**:
  $$\mu \sim \mathcal{N}(\mu_0, \sigma_0^2), \qquad \log\sigma \sim \mathcal{N}\left(\log s, 1.0^2\right)$$
* **Likelihood**:
  $$p(y \mid \mu, \sigma) = \prod_{i=1}^n \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(y_i - \mu)^2}{2\sigma^2}\right)$$
* **Joint Log-Posterior**:
  $$\ln p(\mu, \log\sigma \mid y) = \ln p(\mu) + \ln p(\log\sigma) + \sum_{i=1}^n \ln \mathcal{N}(y_i \mid \mu, \sigma) + \text{const}$$

We implement the **Metropolis-Hastings algorithm** from scratch to sample 30,000 draws from the true joint posterior.



In [ ]:
# MCMC Sampling: Metropolis-Hastings for (mu, log_sigma)
log_joint_posterior <- function(params) {
    mu_curr <- params[1]
    log_s_curr <- params[2]
    s_curr <- exp(log_s_curr)
    
    # 1. Log-Priors
    lp_mu <- dnorm(mu_curr, mean = prior_mu, sd = prior_sigma, log = TRUE)
    lp_sigma <- dnorm(log_s_curr, mean = log(sd(y)), sd = 1.0, log = TRUE)
    
    # 2. Log-Likelihood
    ll <- sum(dnorm(y, mean = mu_curr, sd = s_curr, log = TRUE))
    
    return(lp_mu + lp_sigma + ll)
}

# Run Metropolis-Hastings
set.seed(42)
n_mcmc_samples <- 25000
burn_in <- 5000

curr_params <- c(y_bar, log(sd(y)))
curr_log_prob <- log_joint_posterior(curr_params)

chain_mu <- numeric(n_mcmc_samples)
chain_log_sigma <- numeric(n_mcmc_samples)
accepted <- 0

proposal_scale <- c(0.75, 0.05)

for (step in 1:n_mcmc_samples) {
    proposal <- rnorm(2, mean = curr_params, sd = proposal_scale)
    prop_log_prob <- log_joint_posterior(proposal)
    
    alpha_acc <- prop_log_prob - curr_log_prob
    if (log(runif(1)) < alpha_acc) {
        curr_params <- proposal
        curr_log_prob <- prop_log_prob
        accepted <- accepted + 1
    }
    
    chain_mu[step] <- curr_params[1]
    chain_log_sigma[step] <- curr_params[2]
}

# Retain post burn-in draws
samples_mu <- chain_mu[(burn_in + 1):n_mcmc_samples]
samples_sigma <- exp(chain_log_sigma[(burn_in + 1):n_mcmc_samples])
acceptance_rate <- accepted / n_mcmc_samples

cat("==================================================
")
cat("      MCMC SAMPLING DIAGNOSTICS & SUMMARY         
")
cat("==================================================
")
cat(sprintf("Total Iterations                  : %d
", n_mcmc_samples))
cat(sprintf("Burn-in Discarded                 : %d
", burn_in))
cat(sprintf("Retained Posterior Draws          : %d
", length(samples_mu)))
cat(sprintf("Metropolis Acceptance Rate        : %.2f%% (Optimal: ~23-40%%)
", acceptance_rate * 100))
cat("==================================================
")


---

## Part 4: Bayesian Decision Analysis — Point Estimates, HDI, and ROPE

In Bayesian analysis, the posterior distribution is not just a summary—it is an **engine for making optimal decisions under uncertainty**.

### 4.1 Optimal Point Estimates via Loss Functions
Frequentists default to the sample mean $\bar{x}$. In Bayesian decision theory, the "best" point estimate $\hat{\theta}$ depends explicitly on the cost of being wrong (the **Loss Function**):
1. **Squared Error Loss** $L(\theta, \hat{\theta}) = (\theta - \hat{\theta})^2$:
   Minimizing expected loss yields the **Posterior Mean**:
   $$\hat{\theta} = \mathbb{E}[\mu \mid y] = \frac{1}{S}\sum_{s=1}^S \mu^{(s)}$$
2. **Absolute Error Loss** $L(\theta, \hat{\theta}) = |\theta - \hat{\theta}|$:
   Minimizing expected loss yields the **Posterior Median**:
   $$\hat{\theta} = \text{Median}(\mu \mid y)$$
3. **0-1 Loss** (All errors equally penalized):
   Yields the **Maximum A Posteriori (MAP)** mode.

### 4.2 The 95% Highest Density Interval (HDI)
Unlike an equal-tailed credible interval (which cuts 2.5% off each tail), the **Highest Density Interval (HDI)** possesses two unique mathematical properties:
1. Every point inside the HDI has a **higher posterior probability density** than any point outside.
2. It is the **narrowest possible interval** containing 95% of the posterior mass.

### 4.3 Practical Decision Making: Direct Probability & ROPE
Suppose our Service Level Objective (SLO) requires that mean latency be no greater than $145.0\text{ ms}$.
In the frequentist world, we can only compute a convoluted $p$-value under a hypothetical null.
In the Bayesian world, we ask the question directly:
$$P(\mu \le 145.0\text{ ms} \mid y) = \frac{1}{S}\sum_{s=1}^S \mathbb{I}\left(\mu^{(s)} \le 145.0\right)$$

In addition to one-sided SLO thresholds, Bayesian analysis replaces brittle point-null tests ($H_0: \mu = 140.0\text{ ms}$) with a **Region of Practical Equivalence (ROPE)**:
* **Definition**: A **ROPE** $[\mu_0 - \Delta, \mu_0 + \Delta]$ (for example, $[138.0, 142.0]\text{ ms}$ around a $140.0\text{ ms}$ baseline) is the zone of operational indifference where deviations are too small to matter in production.
* **The HDI + ROPE Decision Rule**:
  1. **Accept Practical Equivalence**: 95% HDI lies *entirely inside* the ROPE.
  2. **Reject Practical Equivalence**: 95% HDI lies *entirely outside* the ROPE.
  3. **Inconclusive**: 95% HDI overlaps the ROPE boundary (collect more data).
*(See **[Chapter 11](11_t_test_frequentist_vs_bayesian.ipynb)** for the complete two-sample BEST + ROPE framework.)*


In [ ]:
# Compute Bayesian Point Estimates & 95% Highest Density Interval (HDI)
compute_hdi <- function(samples, cred_mass = 0.95) {
    sorted_s <- sort(samples)
    n_samples <- length(sorted_s)
    ci_index <- floor(cred_mass * n_samples)
    n_ci <- n_samples - ci_index
    ci_widths <- sorted_s[(ci_index + 1):n_samples] - sorted_s[1:n_ci]
    min_idx <- which.min(ci_widths)
    return(c(sorted_s[min_idx], sorted_s[min_idx + ci_index]))
}

post_mean <- mean(samples_mu)
post_median <- median(samples_mu)
hdi_bounds <- compute_hdi(samples_mu, cred_mass = 0.95)
eti_bounds <- quantile(samples_mu, probs = c(0.025, 0.975))

# Decision Analysis: Probability SLA < 145 ms
prob_meets_slo <- mean(samples_mu <= 145.0)

cat("==================================================
")
cat("     BAYESIAN POINT & INTERVAL ESTIMATES (mu)     
")
cat("==================================================
")
cat(sprintf("Posterior Mean (L2 loss)          : %.3f ms
", post_mean))
cat(sprintf("Posterior Median (L1 loss)        : %.3f ms
", post_median))
cat(sprintf("Posterior Std Dev (Std Error)     : %.3f ms
", sd(samples_mu)))
cat(sprintf("95%% Highest Density Interval (HDI): [%.3f, %.3f] ms (Width: %.3f)
", 
            hdi_bounds[1], hdi_bounds[2], hdi_bounds[2] - hdi_bounds[1]))
cat(sprintf("95%% Equal-Tailed Interval (ETI)   : [%.3f, %.3f] ms (Width: %.3f)
", 
            eti_bounds[1], eti_bounds[2], eti_bounds[2] - eti_bounds[1]))
cat("--------------------------------------------------
")
cat(sprintf("Direct Probability P(mu <= 145 ms): %.2f%%
", prob_meets_slo * 100))
cat("==================================================
")


---

## Part 5: The Posterior Predictive Distribution — Simulating Future Requests

In production engineering, management rarely asks only about the abstract mathematical parameter $\mu$. They ask:
* *"What is the probability that tomorrow's request will time out (> 180 ms)?"*
* *"What latency distribution should our downstream client services design for?"*

### 5.1 The Flaw of Frequentist Plug-In Prediction
A naive frequentist prediction plugs in point estimates: $\tilde{y} \sim \mathcal{N}(\bar{x}, s^2)$.
This **ignores parameter uncertainty**—it pretends we know the exact mean and variance, which systematically under-estimates tail risks.

### 5.2 The Bayesian Posterior Predictive Integral
The Bayesian approach integrates over all parameter uncertainty in the joint posterior:
$$p(\tilde{y} \mid y) = \iint p(\tilde{y} \mid \mu, \sigma) \, p(\mu, \sigma \mid y) \, d\mu \, d\sigma$$

In MCMC, computing this is trivial: for every posterior sample pair $(\mu^{(s)}, \sigma^{(s)})$, we simulate one future observation:
$$\tilde{y}^{(s)} \sim \mathcal{N}\left(\mu^{(s)}, \sigma^{(s)}\right)$$
The resulting collection $\{\tilde{y}^{(1)}, \dots, \tilde{y}^{(S)}\}$ captures both **aleatoric uncertainty** (inherent system randomness $\sigma$) and **epistemic uncertainty** (our imperfect knowledge of $\mu$ and $\sigma$).



In [ ]:
# Simulate Posterior Predictive Distribution (Future Individual Request Latencies)
set.seed(42)
pred_y <- rnorm(length(samples_mu), mean = samples_mu, sd = samples_sigma)

# Tail risk analysis: Timeout SLA = 180 ms
timeout_sla <- 180.0
prob_timeout <- mean(pred_y > timeout_sla)
pred_hdi <- compute_hdi(pred_y, cred_mass = 0.95)

cat("==================================================
")
cat("      POSTERIOR PREDICTIVE RISK ANALYSIS          
")
cat("==================================================
")
cat(sprintf("Simulated Future Requests         : %d
", length(pred_y)))
cat(sprintf("Predicted 95%% Interval of Latency : [%.2f, %.2f] ms
", pred_hdi[1], pred_hdi[2]))
cat(sprintf("Predicted Failure Rate (>180 ms)  : %.2f%%
", prob_timeout * 100))
cat("==================================================
")


---

## Part 6: Comprehensive Multi-Panel Visualization Dashboard

We synthesize the complete Bayesian workflow into a 4-panel publication-grade dashboard ($15 \times 11$, 120 DPI):
1. **The Normal Conjugate Tug-of-War**: Overlays Prior, Likelihood, and Posterior curves to illustrate how precision weighting pulls the posterior between prior belief and sample data.
2. **Joint Posterior Distribution of $(\mu, \sigma)$**: 2D KDE contour map displaying joint uncertainty and marginal distributions.
3. **Posterior of Mean $\mu$ with 95% HDI & SLO**: Posterior density of $\mu$, marking the 95% HDI bar and the shaded probability region satisfying our $145\text{ ms}$ SLO.
4. **Posterior Predictive Distribution vs. Raw Data**: Compares observed request durations against simulated future requests, highlighting the $> 180\text{ ms}$ tail risk.



In [ ]:
# Multi-Panel Visualization Dashboard in Base R
par(mfrow = c(2, 2), mar = c(4.5, 4.5, 3.5, 1.5))

# Panel 1: Gaussian Conjugate Tug-of-War
x_axis <- seq(120, 180, length.out = 400)
p_prior <- dnorm(x_axis, mean = prior_mu, sd = prior_sigma)
p_lik <- dnorm(x_axis, mean = y_bar, sd = sigma_known / sqrt(n))
p_post <- dnorm(x_axis, mean = post_mu, sd = post_std)

plot(x_axis, p_prior, type = "l", col = "#ff7f0e", lwd = 2, lty = 2, ylim = c(0, max(p_post) * 1.1),
     main = "1. Analytical Gaussian Conjugacy", xlab = "Latency (ms)", ylab = "Density", cex.main = 1.1)
lines(x_axis, p_lik, col = "#2ca02c", lwd = 2, lty = 3)
lines(x_axis, p_post, col = "#1f77b4", lwd = 2.5)
legend("topleft", legend = c("Prior N(150, 20)", "Data Likelihood", "Posterior N(mu, sigma)"),
       col = c("#ff7f0e", "#2ca02c", "#1f77b4"), lty = c(2, 3, 1), lwd = c(2, 2, 2.5), bty = "n", cex = 0.8)

# Panel 2: Joint MCMC Draws
sub_idx <- seq(1, length(samples_mu), by = 5)
plot(samples_mu[sub_idx], samples_sigma[sub_idx], col = rgb(0.12, 0.47, 0.71, 0.15), pch = 20,
     main = "2. Joint Posterior Samples p(mu, sigma | y)", xlab = "Mean Latency mu (ms)", ylab = "Std Dev sigma (ms)", cex.main = 1.1)
points(post_mean, mean(samples_sigma), col = "#d62728", pch = 19, cex = 1.5)
legend("topright", legend = c("MCMC Draws", "Posterior Mean"), col = c("#1f77b4", "#d62728"),
       pch = c(20, 19), bty = "n", cex = 0.85)

# Panel 3: Posterior Distribution & 95% HDI
hist(samples_mu, breaks = 40, prob = TRUE, col = "#e6f2ff", border = "#1f77b4",
     main = "3. Posterior Marginal p(mu | y) & 95% HDI", xlab = "Mean Latency (ms)", ylab = "Density", cex.main = 1.1)
lines(density(samples_mu), col = "#1f77b4", lwd = 2)
lines(c(hdi_bounds[1], hdi_bounds[2]), c(0.04, 0.04), col = "#d62728", lwd = 5)
abline(v = 145.0, col = "black", lty = 2, lwd = 2)
legend("topright", legend = c(sprintf("95%% HDI [%.1f, %.1f]", hdi_bounds[1], hdi_bounds[2]), "SLO Threshold (145 ms)"),
       col = c("#d62728", "black"), lty = c(1, 2), lwd = c(4, 2), bty = "n", cex = 0.8)

# Panel 4: Posterior Predictive vs SLA
hist(pred_y, breaks = 45, prob = TRUE, col = "#e8f5e9", border = "#2ca02c",
     main = "4. Posterior Predictive vs SLA Timeout", xlab = "Latency (ms)", ylab = "Density", cex.main = 1.1)
lines(density(pred_y), col = "#2ca02c", lwd = 2)
abline(v = timeout_sla, col = "#d62728", lwd = 2.5, lty = 2)
legend("topright", legend = c(sprintf("Predicted Requests (P(>180ms)=%.1f%%)", prob_timeout * 100), "Timeout SLA (180 ms)"),
       col = c("#2ca02c", "#d62728"), lty = c(1, 2), lwd = c(2, 2.5), bty = "n", cex = 0.8)


---

## Part 7: Robust Bayesian Mean with Student-$t$ Likelihood

What happens when real-world production metrics experience occasional extreme anomalies (e.g., stop-the-world JVM garbage collection, cloud node migrations, or TCP retry storms)?

### 7.1 The Brittleness of Gaussian Likelihood
Under a Gaussian likelihood, the log-likelihood penalty scales with the **square of the error**:
$$\ln p(y_i \mid \mu, \sigma) \propto -\frac{(y_i - \mu)^2}{2\sigma^2}$$
A single outlier at $380\text{ ms}$ exerts immense mechanical leverage, dragging the estimated mean $\mu$ sharply upwards.

### 7.2 The Robust Student-$t$ Likelihood
In the Bayesian paradigm, we solve this by replacing the Normal likelihood with a **Student-$t$ distribution** with $\nu$ degrees of freedom:
$$y_i \sim t(\mu, \sigma, \nu)$$
Because Student's $t$ has polynomial (fat) tails rather than exponential tails, extreme outliers are treated as plausible tail realizations rather than pulling the location parameter $\mu$.



In [ ]:
# Demonstration: Robust Bayesian Mean vs Gaussian under Outlier Contamination
y_corrupted <- c(y, 380.0)

# Robust Student-t Likelihood log-posterior (nu = 4)
log_post_robust <- function(params) {
    mu_curr <- params[1]; log_s_curr <- params[2]; s_curr <- exp(log_s_curr)
    lp_mu <- dnorm(mu_curr, mean = prior_mu, sd = prior_sigma, log = TRUE)
    lp_sigma <- dnorm(log_s_curr, mean = log(sd(y_corrupted)), sd = 1.0, log = TRUE)
    # dt with location and scale
    ll <- sum(dt((y_corrupted - mu_curr) / s_curr, df = 4, log = TRUE) - log(s_curr))
    return(lp_mu + lp_sigma + ll)
}

# Run quick Metropolis sampler for robust model
set.seed(42)
curr_r <- c(mean(y_corrupted), log(sd(y_corrupted)))
curr_lp <- log_post_robust(curr_r)
chain_r_mu <- numeric(10000)

for (s in 1:10000) {
    prop <- rnorm(2, mean = curr_r, sd = c(0.7, 0.05))
    plp <- log_post_robust(prop)
    if (log(runif(1)) < plp - curr_lp) {
        curr_r <- prop
        curr_lp <- plp
    }
    chain_r_mu[s] <- curr_r[1]
}

robust_mu_samples <- chain_r_mu[2001:10000]

cat("==================================================
")
cat("   OUTLIER CONTAMINATION TEST (380 ms OUTLIER)   
")
cat("==================================================
")
cat(sprintf("Clean Sample Mean                 : %.2f ms
", y_bar))
cat(sprintf("Corrupted Sample Mean (Frequentist): %.2f ms (Shift: %+.2f ms)
", 
            mean(y_corrupted), mean(y_corrupted) - y_bar))
cat(sprintf("Gaussian Posterior Mean           : %.2f ms
", 
            (tau_0 * prior_mu + (n + 1) * mean(y_corrupted) / sd(y_corrupted)^2) / (tau_0 + (n + 1) / sd(y_corrupted)^2)))
cat(sprintf("Robust Student-t Posterior Mean   : %.2f ms (Resistant to outlier!)
", mean(robust_mu_samples)))
cat("==================================================
")


---

## 🎯 Check Your Understanding: 3-Tier Scaffolding Exercises

### 🟢 Tier 1: Conceptual Verification
1. **Scenario A**: An engineer observes $n = 10$ query latencies and uses an uninformative (flat) prior. The resulting Bayesian 95% Credible Interval is $[92.1, 108.4]\text{ ms}$. Can the engineer legitimately claim: *"There is a 95% probability that the true mean latency lies between 92.1 and 108.4 ms"*?
2. **Scenario B**: What happens to the Bayesian posterior distribution of $\mu$ if the engineer sets prior uncertainty $\sigma_0 \to \infty$ (completely uninformative prior)? How does the posterior compare to the frequentist sampling distribution?
3. **Scenario C**: Why does the Posterior Predictive Distribution for future request latency have a wider standard deviation than the posterior distribution of the mean $\mu$?

<details>
<summary><b>🔍 Click here to view the Tier 1 answers</b></summary>

1. **Answer A**: **Yes!** In the Bayesian framework, probability represents degree of belief conditioned on data. Once the posterior distribution is computed, statements like $P(92.1 \le \mu \le 108.4 \mid y) = 0.95$ are completely valid. (This is in direct contrast to the frequentist paradigm, where such a statement is a fallacy).
2. **Answer B**: As $\sigma_0 \to \infty$, prior precision $\tau_0 \to 0$. The posterior mean converges exactly to the sample mean ($\bar{y}$), and posterior standard deviation converges to the standard error ($s/\sqrt{n}$). The mathematical numbers match the frequentist output, but with a Bayesian interpretation.
3. **Answer C**: Because the Posterior Predictive Distribution incorporates **both** parameter uncertainty (epistemic uncertainty: we do not know the exact $\mu$) **and** physical observation variance (aleatoric uncertainty: individual requests scatter with scale $\sigma$). Its variance is approximately $\sigma^2 + \sigma^2/n$, which is much wider than the uncertainty of the mean alone ($\sigma^2/n$).
</details>



In [ ]:
# ============================================================================
# 🟡 Tier 2: Guided Calculation — Implement `gaussian_conjugate_update()`
# ============================================================================

gaussian_conjugate_update <- function(prior_mean, prior_sd, data, known_sd = NULL, cred_mass = 0.95) {
    n_obs <- length(data)
    y_bar <- mean(data)
    
    if (is.null(known_sd)) {
        sigma_use <- sd(data)
    } else {
        sigma_use <- known_sd
    }
    
    tau_0 <- 1.0 / (prior_sd^2)
    tau_data <- n_obs / (sigma_use^2)
    tau_post <- tau_0 + tau_data
    
    post_mean <- (tau_0 * prior_mean + tau_data * y_bar) / tau_post
    post_sd <- 1.0 / sqrt(tau_post)
    
    alpha <- 1.0 - cred_mass
    z_crit <- qnorm(1.0 - alpha / 2.0)
    
    hdi_lower <- post_mean - z_crit * post_sd
    hdi_upper <- post_mean + z_crit * post_sd
    
    return(list(
        prior_weight = tau_0 / tau_post,
        data_weight = tau_data / tau_post,
        post_mean = post_mean,
        post_sd = post_sd,
        hdi_lower = hdi_lower,
        hdi_upper = hdi_upper
    ))
}

test_res <- gaussian_conjugate_update(prior_mu, prior_sigma, y)
cat("Tier 2 Implementation Test:
")
cat(sprintf("  Posterior Mean = %.3f ms, SD = %.3f ms
", test_res$post_mean, test_res$post_sd))
cat(sprintf("  95%% HDI = [%.3f, %.3f] ms
", test_res$hdi_lower, test_res$hdi_upper))

# Unit assertions
stopifnot(abs(test_res$post_mean - post_mu) < 1e-4)
stopifnot(abs(test_res$post_sd - post_std) < 1e-4)
cat("✅ All Tier 2 assertions passed successfully!
")


<details>
<summary><b>🔍 Click here to view the Tier 2 detailed explanation</b></summary>

```python
# The function implements the exact analytical equations of Gaussian conjugacy:
# 1. tau_0 = 1 / sigma_0^2
# 2. tau_n = n / sigma^2
# 3. tau_post = tau_0 + tau_n
# 4. mu_post = (tau_0 * mu_0 + tau_n * y_bar) / tau_post
# 5. sigma_post = 1 / sqrt(tau_post)
# 6. bounds = mu_post ± z* * sigma_post
```
</details>



---

### 🔴 Tier 3: Production Simulation Challenge — The "Cold-Start" Microservice Problem
> **The Problem**: When rolling out a new microservice region or canary container, initial telemetry is extremely sparse ($n = 3$ requests).
> * **Frequentist Approach**: A Student-$t$ confidence interval with $df = 2$ uses critical value $t_{0.025, 2} \approx 4.303$. The interval explodes to hundreds of milliseconds in width, rendering the SLA check useless.
> * **Bayesian Approach**: Engineering domain knowledge from existing regions provides an informative prior ($\mu_0 = 145\text{ ms}, \sigma_0 = 15\text{ ms}$). Bayesian updating gracefully fuses the 3 new observations with historical prior knowledge, preventing interval explosion.
>
> **Your Task**:
> 1. Simulate 1,000 canary launches with sparse sample size $n = 3$ drawn from $\mathcal{N}(\text{true\_mu} = 142.5, \text{true\_sigma} = 28.0)$.
> 2. Compare the average 95% interval width and Mean Squared Error (MSE) of:
>    * The Frequentist Student-$t$ Estimator ($\bar{x}$).
>    * The Bayesian Conjugate Estimator ($\mu_{\text{post}}$ with prior $\mu_0 = 145, \sigma_0 = 15$).



In [ ]:
# Production Simulation: The Cold-Start Microservice Problem (n = 3)
set.seed(42)
n_canaries <- 1000
n_sparse <- 3

prior_canary_mu <- 145.0
prior_canary_sigma <- 15.0

freq_errors <- numeric(n_canaries)
bayes_errors <- numeric(n_canaries)

for (sim in 1:n_canaries) {
    true_canary_mu <- rnorm(1, mean = 145.0, sd = 10.0)
    y_sparse <- rnorm(n_sparse, mean = true_canary_mu, sd = 25.0)
    
    # Frequentist: sample mean
    m_freq <- mean(y_sparse)
    
    # Bayesian: conjugate update with prior
    res_bayes <- gaussian_conjugate_update(prior_canary_mu, prior_canary_sigma, y_sparse, known_sd = 25.0)
    m_bayes <- res_bayes$post_mean
    
    freq_errors[sim] <- (m_freq - true_canary_mu)^2
    bayes_errors[sim] <- (m_bayes - true_canary_mu)^2
}

rmse_freq <- sqrt(mean(freq_errors))
rmse_bayes <- sqrt(mean(bayes_errors))

cat("====================================================================
")
cat("      TIER 3 SIMULATION: COLD-START ESTIMATION (n = 3)              
")
cat("====================================================================
")
cat(sprintf("Frequentist Sample Mean RMSE      : %.2f ms
", rmse_freq))
cat(sprintf("Bayesian Conjugate Posterior RMSE : %.2f ms (Error reduction: %.1f%%)
", 
            rmse_bayes, 100 * (1 - rmse_bayes / rmse_freq)))
cat("====================================================================
")
cat("Conclusion: When data is sparse (n=3), Bayesian shrinkage against prior domain
")
cat("knowledge cuts estimation error substantially compared to naive sample means.
")


<details>
<summary><b>🔍 Click here to view the Tier 3 solution & architectural takeaway</b></summary>

### The Takeaway for Production Reliability:
1. **The Curse of Small Samples in Frequentist Stats**: When $n \le 3$, sample standard deviation $s$ fluctuates wildly. The $t$-distribution critical value ($t_{0.025, 2} \approx 4.303$) penalizes small samples so severely that confidence intervals become hundreds of milliseconds wide, making automated canary decisions impossible.
2. **Prior as Regularization**: An informative prior acts as an anchor (mathematically equivalent to $L_2$ regularization or shrinkage). It prevents extreme small-sample noise from triggering false canary rollbacks.
3. **Graceful Transition**: As canary traffic scales ($n = 3 \to 30 \to 3,000$), the Bayesian estimator automatically sheds its prior reliance and converges seamlessly to the empirical data evidence.
</details>



---

## 🏁 Summary Checklist: How to Calculate a Mean in a Bayesian World

```
                 Do you have known variance σ and Gaussian data?
                                   /             \
                                  /               \
                               [YES]             [NO]
                                 |                 |
                       Exact Analytical      Are there extreme outliers
                      Gaussian Conjugacy      or heavy-tailed spikes?
                                                   /           \
                                                  /             \
                                            [Outliers]       [Standard]
                                                |                 |
                                        Robust Student-t      Joint MCMC
                                           Likelihood          (μ, σ)
```

### Essential Bayesian Formula Card:
* **Prior Precision**: $\tau_0 = \frac{1}{\sigma_0^2}$
* **Data Precision**: $\tau_n = \frac{n}{\sigma^2}$
* **Posterior Precision**: $\tau_{\text{post}} = \tau_0 + \tau_n \implies \sigma_{\text{post}} = \frac{1}{\sqrt{\tau_0 + \tau_n}}$
* **Posterior Mean**: $\mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_n \bar{y}}{\tau_0 + \tau_n}$
* **95% Highest Density Interval (HDI)**: The narrowest interval containing 95% of posterior draws.
* **Posterior Predictive Simulation**: Draw $\tilde{y} \sim \mathcal{N}\left(\mu^{(s)}, \sigma^{(s)}\right)$ for each posterior sample.


---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 03: Calculating a Mean (Frequentist)](03_calculating_a_mean_frequentist.ipynb)** *(Part I: Foundations, Conjugacy & Estimating a Mean)*
* ➡️ **Next**: **[Ch 05: 2D Grid Approximation](05_frequentist_vs_grid_approximation.ipynb)** *(Part II: Numerical & Sampling Engines)*
* 🔄 **Companion Tracks for Chapter 04**: [📖 Conceptual](../conceptual/04_calculating_a_mean_bayesian.ipynb) | [🤖 AI-Native](../ai/04_calculating_a_mean_bayesian.ipynb) | [🐍 Python](../python/04_calculating_a_mean_bayesian.ipynb) | [🔵 R](../r/04_calculating_a_mean_bayesian.ipynb)
